# 04 — Tracking de Objetos

## Detección vs. Tracking

Detectar objetos en video es como revisar cada foto de una cámara de seguridad:
sabes cuántos objetos hay en ese instante, pero no sabes si el objeto
del frame 1 es el mismo que el del frame 2.

**El tracker resuelve eso:**
actúa como un guardia con lista de asistencia — compara los objetos nuevos
con los del frame anterior y asigna el mismo número de credencial si los reconoce.

```
Frame N:   detecciones sin ID  ──► tracker ──► detecciones con tracker_id
Frame N+1: detecciones sin ID  ──► tracker ──► mismos tracker_id (si mismo objeto)
```

In [1]:
import supervision as sv
from ultralytics import YOLO
import cv2
import numpy as np
import urllib.request

print("Descargando video de muestra (puede tardar un momento)...")
urllib.request.urlretrieve(
    "https://media.roboflow.com/supervision/video-examples/vehicles.mp4",
    "../../assets/curso/videos/vehicles.mp4"
)
print("Video listo.")

video_info = sv.VideoInfo.from_video_path(
    "../../assets/curso/videos/vehicles.mp4"
)

print(f"\nResolución: {video_info.width} × {video_info.height} px")
print(f"FPS: {video_info.fps}")
print(f"Duración: {video_info.total_frames / video_info.fps:.1f} segundos")

Descargando video de muestra (puede tardar un momento)...
Video listo.

Resolución: 3840 × 2160 px
FPS: 25.0
Duración: 21.5 segundos


## ¿Por qué usamos `sv.ByteTrack` en este notebook?

`sv.ByteTrack` es el tracker integrado en Supervision. Su ventaja en este notebook
es la API directa: `update_with_detections(det)` acepta y devuelve `sv.Detections`
sin pasos intermedios — ideal para aprender el concepto de tracking.

A partir de Supervision v0.28.0, `sv.ByteTrack` está marcado como **deprecated**
y será removido en v0.30.0, en favor del paquete externo
[`trackers`](https://pypi.org/project/trackers/). La razón del cambio es separar
la lógica de tracking de la librería principal y estandarizar la API entre trackers.

**¿Por qué no migramos aquí?** Porque el objetivo de este notebook es entender
los *conceptos*: qué es un `tracker_id`, cómo persiste entre frames y cómo se
combina con los annotators. `sv.ByteTrack` es suficiente para aprender eso con
menos fricción — la API es simple y el comportamiento es idéntico.

En **NB05** (Zonas) y **NB09** (Video + SAM) usamos el paquete `trackers`
actualizado, ya que en esos notebooks es importante trabajar con la API vigente.

In [2]:
model = YOLO("../../assets/curso/models/yolov8n.pt")

# sv.ByteTrack es la API clásica de Supervision — funciona pero está siendo reemplazada.
tracker = sv.ByteTrack()

# Inspeccionamos un solo frame para ver el estado ANTES del tracker
cap = cv2.VideoCapture("../../assets/curso/videos/vehicles.mp4")
ret, primer_frame = cap.read()
cap.release()

results_test = model(primer_frame, verbose=False)[0]
det_sin_tracker = sv.Detections.from_ultralytics(results_test)

print("ANTES de pasar por el tracker:")
print(f"  tracker_id: {det_sin_tracker.tracker_id}")

det_con_tracker = tracker.update_with_detections(det_sin_tracker)

print("\nDESPUÉS de pasar por el tracker:")
print(f"  tracker_id: {det_con_tracker.tracker_id}")

ANTES de pasar por el tracker:
  tracker_id: None

DESPUÉS de pasar por el tracker:
  tracker_id: [1 2 3 4]


## El pipeline completo con tracking

In [3]:
# Reiniciamos el tracker para que los IDs empiecen desde 1
tracker.reset()

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()
trace_annotator = sv.TraceAnnotator()

def procesar_frame(frame: np.ndarray, frame_idx: int) -> np.ndarray:
    results = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)

    detections = tracker.update_with_detections(detections)

    labels = [f"ID:{tid}" for tid in detections.tracker_id]

    annotated = box_annotator.annotate(
        scene=frame.copy(),
        detections=detections
    )

    annotated = label_annotator.annotate(
        scene=annotated,
        detections=detections,
        labels=labels
    )

    annotated = trace_annotator.annotate(
        scene=annotated,
        detections=detections
    )

    return annotated

sv.process_video(
    source_path="../../assets/curso/videos/vehicles.mp4",
    target_path="../../assets/curso/videos/vehicles_tracked.mp4",
    callback=procesar_frame
)

print("Video guardado: ../../assets/curso/videos/vehicles_tracked.mp4")

Video guardado: ../../assets/curso/videos/vehicles_tracked.mp4


## Exploración interactiva

### Experimento 1: Inspeccionar tracker_id frame a frame

In [4]:
tracker.reset()

cap = cv2.VideoCapture("../../assets/curso/videos/vehicles.mp4")

for frame_num in range(3):
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)[0]
    det = sv.Detections.from_ultralytics(results)
    det = tracker.update_with_detections(det)

    print(f"Frame {frame_num}: {len(det)} objetos | IDs: {det.tracker_id}")

cap.release()

Frame 0: 4 objetos | IDs: [1 2 3 4]
Frame 1: 4 objetos | IDs: [1 3 2 4]
Frame 2: 3 objetos | IDs: [1 3 2]


### Experimento 2: Mostrar clase e ID al mismo tiempo

In [5]:
tracker.reset()

def callback_clase_id(frame: np.ndarray, _: int) -> np.ndarray:
    results = model(frame, verbose=False)[0]

    det = sv.Detections.from_ultralytics(results)
    det = tracker.update_with_detections(det)

    labels = [
        f"{results.names[c]} #{tid}"
        for c, tid in zip(det.class_id, det.tracker_id)
    ]

    scene = box_annotator.annotate(
        scene=frame.copy(),
        detections=det
    )

    return label_annotator.annotate(
        scene=scene,
        detections=det,
        labels=labels
    )

sv.process_video(
    source_path="../../assets/curso/videos/vehicles.mp4",
    target_path="../../assets/curso/videos/vehicles_clase_id.mp4",
    callback=callback_clase_id
)

print("Guardado: ../../assets/curso/videos/vehicles_clase_id.mp4")

Guardado: ../../assets/curso/videos/vehicles_clase_id.mp4


### Experimento 3: Filtrar antes de trackear

In [6]:
CLASE_OBJETIVO = 2  # car en COCO

tracker.reset()

def callback_solo_autos(frame: np.ndarray, _: int) -> np.ndarray:
    results = model(frame, verbose=False)[0]

    det = sv.Detections.from_ultralytics(results)

    det = det[det.class_id == CLASE_OBJETIVO]

    det = tracker.update_with_detections(det)

    labels = [f"auto #{tid}" for tid in det.tracker_id]

    scene = box_annotator.annotate(
        scene=frame.copy(),
        detections=det
    )

    return label_annotator.annotate(
        scene=scene,
        detections=det,
        labels=labels
    )

sv.process_video(
    source_path="../../assets/curso/videos/vehicles.mp4",
    target_path="../../assets/curso/videos/vehicles_autos.mp4",
    callback=callback_solo_autos
)

print("Guardado: ../../assets/curso/videos/vehicles_autos.mp4")

Guardado: ../../assets/curso/videos/vehicles_autos.mp4


## 🚀 Reto de extensión

**Tarea:** Muestra en la etiqueta cuántos frames lleva visible cada objeto.

**Pista:** Crea un diccionario fuera del callback para contar frames por ID:
```python
frame_count = {}

def mi_callback(frame, _):
    # ... detección y tracking ...
    for tid in detections.tracker_id:
        frame_count[tid] = frame_count.get(tid, 0) + 1
    labels = [f"#{tid} ({frame_count[tid]}f)" for tid in detections.tracker_id]
    # ... anotar y retornar ...
```

In [7]:
frame_count = {}

tracker.reset()

def mi_callback(frame: np.ndarray, _: int) -> np.ndarray:
    results = model(frame, verbose=False)[0]
    det = sv.Detections.from_ultralytics(results)
    det = tracker.update_with_detections(det)

    for tid in det.tracker_id:
        frame_count[tid] = frame_count.get(tid, 0) + 1

    labels = [
        f"#{tid} ({frame_count[tid]}f)"
        for tid in det.tracker_id
    ]

    scene = box_annotator.annotate(
        scene=frame.copy(),
        detections=det
    )

    scene = label_annotator.annotate(
        scene=scene,
        detections=det,
        labels=labels
    )

    return scene

sv.process_video(
    source_path="../../assets/curso/videos/vehicles.mp4",
    target_path="../../assets/curso/videos/vehicles_frames.mp4",
    callback=mi_callback
)